In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count, countDistinct

In [0]:
# Read the arxiv_bronze table from Unity Catalog
df_whole = spark.table("scientific_trend_analysis.lds.arxiv_bronze")
display(df_whole.limit(5))

In [0]:
# 1. Data Structure & Types
print("\n1. Data Structure & Column Types")
print("-" * 100)
total_rows = df_whole.count()
columns = df_whole.columns
print(f"Total records: {total_rows:,}")
print(f"Total columns: {len(columns)}\n")
print("Column Names and Data Types:")
df_whole.printSchema()


# 2. Missing Values Distribution
print("\n2. Missing Values Distribution")
print("-" * 100)
missing_stats = df_whole.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in columns
])

missing_df = missing_stats.toPandas().T
missing_df.columns = ['missing_count']
missing_df['missing_percentage[%]'] = (missing_df['missing_count'] / total_rows * 100).round(1)
missing_df = missing_df.sort_values('missing_count', ascending=False)

print(f"Columns with missing values:\n")
print(missing_df[missing_df['missing_count'] > 0])
print(f"\nColumns with no missing values:")
print(missing_df[missing_df['missing_count'] == 0].index.tolist())


# 3. Duplicate Records
print("\n3. Duplicate Records Analysis")
print("-" * 100)

# Check for duplicate IDs
if 'id' in columns:
    id_stats = df_whole.agg(
        count('id').alias('id_total'),
        countDistinct('id').alias('id_unique')
    ).collect()[0]
    
    total_ids = id_stats['id_total']
    unique_ids = id_stats['id_unique']
    duplicate_ids = total_ids - unique_ids
    print(f"Duplicate ids: {duplicate_ids:,}")

# Check for duplicate title+abstract combinations
if 'title' in columns and 'abstract' in columns:
    dup_count = df_whole.groupBy('title', 'abstract').count().filter(col('count') > 1).count()
    print(f"Duplicate title + abstract combinations: {dup_count:,}")